# LLM Context Sentence Generation

This notebook generates controlled Turkish contextual sentences for the unique target words used in the thesis project. The generated sentences are intended for contextual embedding experiments where each target word must appear as an unchanged lexical item in a short, natural sentence.

The notebook uses OpenCode Go through an OpenAI-compatible API endpoint, with Qwen 3.6 Plus as the default generation model. It writes raw model responses after each batch so generation can be resumed without repeating completed API calls.

## Methodological note

For each target word, the prompt asks the language model to produce exactly ten Turkish sentences. The instructions require the target word to appear exactly once and exactly as written, without inflection, suffixation, spelling changes, or capitalization changes. This constraint is important for the later embedding analysis because it keeps the surface form of the target word stable across isolated and contextual conditions.

Generation is performed in batches of ten target words. After each batch, the raw model response is appended to a JSONL file. This makes the procedure auditable and allows interrupted runs to resume safely: batches already present in the raw-response file for the same provider, model, prompt version, and word list are skipped. A separate validation step parses the raw JSON, checks exact target-word matches with a Turkish-aware regular expression, measures sentence length, flags duplicate sentences within each target word, and exports both valid and invalid rows.

## Setup

Install the OpenAI Python SDK in the active notebook environment before running the generation cells:

```bash
pip install -U openai
```

The OpenCode Go API key is read from `OPENCODE_API_KEY`. The notebook also loads a local `.env` file from the project root, so a line such as `OPENCODE_API_KEY="..."` is sufficient. The key is intentionally not stored in the notebook.

In [20]:
from __future__ import annotations

import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

try:
    from openai import OpenAI
except ImportError as exc:
    raise ImportError(
        "Install the OpenAI Python SDK before running this notebook: "
        "pip install -U openai"
    ) from exc


# This notebook is meant to run from the project root.
PROJECT_ROOT = Path.cwd()

# Load simple KEY=VALUE pairs from .env without printing secret values.
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


load_env_file(PROJECT_ROOT / ".env")

# Input and output file paths.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
INPUT_PATH = TABLES_DIR / "0105-tokenization_unique_words.csv"
OUTPUT_PATH = PROCESSED_DIR / "context_sentences_llm.csv"
RAW_PATH = PROCESSED_DIR / "context_sentences_llm_raw.jsonl"
INVALID_PATH = PROCESSED_DIR / "context_sentences_llm_invalid.csv"

# OpenCode Go API settings.
LLM_PROVIDER = "opencode"
OPENCODE_BASE_URL = "https://opencode.ai/zen/go/v1"
GENERATION_MODEL = os.environ.get("OPENCODE_MODEL", "deepseek-v4-flash")

# Generation settings.
BATCH_SIZE = 10
SENTENCES_PER_WORD = 10
PROMPT_VERSION = "tr_context_sentences_v2_opencode"
REQUEST_SLEEP_SECONDS = 1.0
MAX_OUTPUT_TOKENS = 12000

# Create the output folder if it does not exist.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_PATH}")
print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {GENERATION_MODEL}")

Project root: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect
Input: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/outputs/tables/0105-tokenization_unique_words.csv
Provider: opencode
Model: deepseek-v4-flash


## Load target words

The input table must contain one row per unique target word. The expected column name is `word`. If the table has only one column, that column is used as the target-word column.

In [21]:
def load_target_words(path: Path) -> list[str]:
    """Load unique target words from the configured CSV file."""
    # First check that the expected input file exists.
    if not path.exists():
        raise FileNotFoundError(
            f"Expected input file not found: {path}\n"
            "Create this file before generation so the pipeline has a stable input path."
        )

    # Read the CSV file and find the word column.
    df = pd.read_csv(path)
    if "word" in df.columns:
        word_col = "word"
    elif len(df.columns) == 1:
        word_col = df.columns[0]
    else:
        raise ValueError(
            f"Could not infer target-word column from columns: {list(df.columns)}"
        )

    # Remove empty values and duplicate words.
    words = (
        df[word_col]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
        .tolist()
    )
    if not words:
        raise ValueError(f"No target words found in {path}")
    return words


target_words = load_target_words(INPUT_PATH)
print(f"Loaded {len(target_words):,} target words.")
target_words[:10]

Loaded 317 target words.


['adalet',
 'affedişi',
 'ahşap',
 'aile',
 'akıllı',
 'alacaklanmak',
 'alet',
 'alkol',
 'almışlardır',
 'alışkın']

## Prompt and response format

The prompt states the linguistic constraints and explicitly describes the JSON structure expected from the model. The API call also requests a JSON object response when the OpenAI-compatible endpoint supports `response_format`.

In [23]:
# Expected JSON structure for each model response.
EXPECTED_JSON_EXAMPLE = {
    "items": [
        {
            "target_word": "örnek",
            "sentences": [
                "Bugün sınıfta örnek üzerinden kısa bir tartışma yaptık."
            ],
        }
    ]
}


def build_prompt(words: list[str]) -> str:
    """Build the batch prompt for a list of target words."""
    # Put the words in the prompt as a numbered list.
    numbered_words = "\n".join(f"{i + 1}. {word}" for i, word in enumerate(words))
    json_template = json.dumps(EXPECTED_JSON_EXAMPLE, ensure_ascii=False, indent=2)

    prompt = f"""
You are generating controlled Turkish context sentences for a semantic similarity thesis dataset.

Target words:
{numbered_words}

For each target word, generate exactly {SENTENCES_PER_WORD} natural Turkish sentences.
Return exactly {SENTENCES_PER_WORD} strings in the "sentences" array for each target_word.
Each sentence must contain the target word exactly as written.
Do not inflect the target word.
Do not add suffixes to the target word.
Do not change spelling or capitalization.
Do not place the target word at the beginning of a sentence unless its capitalization is already correct.
The target word must appear exactly once.
Sentences should be grammatical Turkish.
Sentences should be 8-14 words long.
Use varied but plausible contexts.
Keep the same basic sense of the word across sentences.
Avoid dictionary definitions, poetry, idioms, lists, and artificial templates.

Return only valid JSON with this exact structure:
{json_template}
"""
    return prompt.strip()


def parse_json_text(text: str) -> dict:
    """Parse the model text as JSON and fail loudly if the response is not valid JSON."""
    # Remove Markdown fences if the model ignores the JSON-only instruction.
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    return json.loads(cleaned)


def response_metadata(response) -> dict:
    """Extract lightweight response metadata from an OpenAI-compatible response."""
    # Store basic details such as token usage and finish reason.
    metadata = {}
    usage = getattr(response, "usage", None)
    if usage is not None:
        if hasattr(usage, "model_dump"):
            metadata["usage"] = usage.model_dump(mode="json")
        else:
            metadata["usage"] = str(usage)
    choices = getattr(response, "choices", None) or []
    if choices:
        metadata["finish_reason"] = str(getattr(choices[0], "finish_reason", ""))
    return metadata


def create_chat_completion(client: OpenAI, prompt: str):
    """Call the OpenAI-compatible chat completions endpoint."""
    # Request JSON mode first; retry without it if the endpoint rejects the parameter.
    request_kwargs = {
        "model": GENERATION_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
        "top_p": 0.95,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "response_format": {"type": "json_object"},
    }

    try:
        return client.chat.completions.create(**request_kwargs)
    except Exception as exc:
        message = str(exc).lower()
        if "response_format" not in message and "json_object" not in message:
            raise
        request_kwargs.pop("response_format")
        return client.chat.completions.create(**request_kwargs)


def generate_batch(client: OpenAI, words: list[str], batch_id: int) -> dict:
    """Call the model for one batch and return a JSONL-serializable record."""
    # Keep the raw response and parsed JSON in the same batch record.
    prompt = build_prompt(words)
    record = {
        "batch_id": batch_id,
        "words": words,
        "provider": LLM_PROVIDER,
        "prompt_version": PROMPT_VERSION,
        "generation_model": GENERATION_MODEL,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "ok": False,
        "raw_text": None,
        "parsed": None,
        "error": None,
    }

    try:
        # OpenCode Go API call.
        response = create_chat_completion(client, prompt)
        raw_text = response.choices[0].message.content
        record["raw_text"] = raw_text
        record["parsed"] = parse_json_text(raw_text)
        record["metadata"] = response_metadata(response)
        record["ok"] = True
    except Exception as exc:
        record["error"] = repr(exc)

    return record


## Resumable generation

This section appends one JSON object per batch to `context_sentences_llm_raw.jsonl`. A batch is considered complete only when the raw file already contains a successful record with the same `batch_id`, provider, model, prompt version, and word list. This means older Gemini records can remain in the same raw file without being treated as completed OpenCode/Qwen batches.

In [24]:
def iter_batches(words: list[str], batch_size: int) -> list[tuple[int, list[str]]]:
    """Return numbered batches using zero-based batch IDs."""
    # Split the word list into batches of 10.
    return [
        (batch_id, words[start : start + batch_size])
        for batch_id, start in enumerate(range(0, len(words), batch_size))
    ]


def read_jsonl(path: Path) -> list[dict]:
    # If the raw response file does not exist, no batches are complete yet.
    if not path.exists():
        return []
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL line {line_number} in {path}") from exc
    return records


def is_current_success(record: dict, expected_batches: dict[int, list[str]]) -> bool:
    """Return True only for successful records from the current provider/model setup."""
    # Old Gemini records should not mark Qwen batches as complete.
    if record.get("ok") is not True or "batch_id" not in record:
        return False
    batch_id = int(record["batch_id"])
    return (
        record.get("provider") == LLM_PROVIDER
        and record.get("generation_model") == GENERATION_MODEL
        and record.get("prompt_version") == PROMPT_VERSION
        and record.get("words") == expected_batches.get(batch_id)
    )


def completed_batch_ids(path: Path, expected_batches: dict[int, list[str]]) -> set[int]:
    # Count only successful batches that match the current configuration.
    return {
        int(record["batch_id"])
        for record in read_jsonl(path)
        if is_current_success(record, expected_batches)
    }


def append_jsonl(record: dict, path: Path) -> None:
    # Append each batch result as a separate JSONL line.
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Resume safely by skipping completed batches.
all_batches = iter_batches(target_words, BATCH_SIZE)
expected_batches = dict(all_batches)
done_batches = completed_batch_ids(RAW_PATH, expected_batches)
pending_batches = [(batch_id, words) for batch_id, words in all_batches if batch_id not in done_batches]

print(f"Total batches: {len(all_batches):,}")
print(f"Completed batches for current model in raw JSONL: {len(done_batches):,}")
print(f"Pending batches: {len(pending_batches):,}")

Total batches: 32
Completed batches for current model in raw JSONL: 0
Pending batches: 32


In [25]:
# Read the API key from the environment; do not write it in the notebook.
api_key = os.environ.get("OPENCODE_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENCODE_API_KEY in .env or in the environment before running generation.")

# Create the OpenCode Go client through the OpenAI-compatible API.
client = OpenAI(api_key=api_key, base_url=OPENCODE_BASE_URL)

for batch_position, (batch_id, words) in enumerate(pending_batches, start=1):
    print(
        f"Generating batch {batch_id} "
        f"({batch_position}/{len(pending_batches)}): {', '.join(words)}"
    )
    # Generate the batch and immediately save it to the raw JSONL file.
    record = generate_batch(client, words, batch_id)
    append_jsonl(record, RAW_PATH)

    if record["ok"]:
        print(f"  saved raw response for batch {batch_id}")
    else:
        print(f"  saved error record for batch {batch_id}: {record['error']}")

    time.sleep(REQUEST_SLEEP_SECONDS)

print("Generation pass complete.")

Generating batch 0 (1/32): adalet, affedişi, ahşap, aile, akıllı, alacaklanmak, alet, alkol, almışlardır, alışkın
  saved error record for batch 0: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
Generating batch 1 (2/32): anarşist, anne, anormalleşiyor, araba, arkadaşımsı, asosyalleştirdikleri, atatürkist, atatürkçülerden, ateistlik, atletik
  saved error record for batch 1: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
Generating batch 2 (3/32): avare, aykırı, barınak, barıştırılırken, bağışlayışı, başladılarsa, başlamadılarsa, başıboş, beyaz, bilimci
  saved error record for batch 2: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
Generating batch 3 (4/32): bilişim, bitirdilerse, bitki, biçimlendirilmişlerdir, bolluklarda, borçlanmak, boza, brüt, böcek, büyücü
  saved error record for batch 3: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
Generating batch 4 (5/32): bıçak, cam, cansız, ceket, cenin, cevher, cezalandırışı, cihaz,

## Parse and validate sentences

The exact-match validation uses a regular expression that treats Turkish letters, digits, underscores, and apostrophes as word-internal characters. This prevents a target word from being counted when it appears inside a longer word or a suffixed form such as `word'a`.

In [ ]:
# Include Turkish characters in word-boundary checks.
TURKISH_WORD_CHARS = "A-Za-z0-9_ÇĞİÖŞÜçğıöşü'’"
WORD_TOKEN_PATTERN = re.compile(r"[A-Za-z0-9_ÇĞİÖŞÜçğıöşü]+(?:['’][A-Za-z0-9_ÇĞİÖŞÜçğıöşü]+)?")


def exact_match_count(sentence: str, target_word: str) -> int:
    # Count the target word only as a separate unsuffixed word.
    pattern = re.compile(
        rf"(?<![{TURKISH_WORD_CHARS}]){re.escape(target_word)}(?![{TURKISH_WORD_CHARS}])"
    )
    return len(pattern.findall(sentence))


def count_words(sentence: str) -> int:
    # Count the words in the sentence.
    return len(WORD_TOKEN_PATTERN.findall(sentence))


def normalize_sentence(sentence: str) -> str:
    # Normalize spacing and case for duplicate checks.
    return re.sub(r"\s+", " ", sentence.strip()).casefold()


def rows_from_raw_records(records: list[dict], expected_batches: dict[int, list[str]]) -> pd.DataFrame:
    # Build table rows from raw JSONL records for the current model only.
    rows = []
    successful_records_by_batch = {}
    for record in records:
        if is_current_success(record, expected_batches):
            successful_records_by_batch[int(record["batch_id"])] = record

    for batch_id in sorted(successful_records_by_batch):
        record = successful_records_by_batch[batch_id]
        generation_model = record.get("generation_model", GENERATION_MODEL)
        prompt_version = record.get("prompt_version", PROMPT_VERSION)
        parsed = record.get("parsed") or {}
        items = parsed.get("items", [])

        # Match the returned items by target word.
        items_by_target = {
            str(item.get("target_word", "")).strip(): item
            for item in items
            if isinstance(item, dict)
        }

        for target_word in record.get("words", []):
            item = items_by_target.get(target_word, {})
            sentences = item.get("sentences", [])
            if not isinstance(sentences, list):
                continue

            for sentence_id, sentence in enumerate(sentences, start=1):
                sentence = str(sentence).strip()
                rows.append(
                    {
                        "target_word": target_word,
                        "sentence_id": sentence_id,
                        "sentence": sentence,
                        "source": LLM_PROVIDER,
                        "generation_model": generation_model,
                        "prompt_version": prompt_version,
                        "batch_id": batch_id,
                    }
                )

    return pd.DataFrame(rows)


raw_records = read_jsonl(RAW_PATH)
sentences_df = rows_from_raw_records(raw_records, expected_batches)

if sentences_df.empty:
    raise ValueError(f"No successful generated sentences found for {GENERATION_MODEL} in {RAW_PATH}")

# Compute validation variables for each sentence.
sentences_df["exact_match_count"] = sentences_df.apply(
    lambda row: exact_match_count(row["sentence"], row["target_word"]), axis=1
)
sentences_df["valid_exact_match"] = sentences_df["exact_match_count"].eq(1)
sentences_df["word_count"] = sentences_df["sentence"].map(count_words)
sentences_df["_normalized_sentence"] = sentences_df["sentence"].map(normalize_sentence)
sentences_df["is_duplicate"] = sentences_df.duplicated(
    subset=["target_word", "_normalized_sentence"], keep="first"
)
sentences_df["valid"] = (
    sentences_df["valid_exact_match"]
    & sentences_df["word_count"].between(8, 18, inclusive="both")
    & ~sentences_df["is_duplicate"]
)

sentences_df = sentences_df.drop(columns=["_normalized_sentence"])
sentences_df.head()

## Export final files and summary

The main CSV contains all parsed generated sentences for the current provider/model and validation flags. The invalid CSV is a filtered copy containing only rows where `valid == False`. The raw JSONL file remains the source-of-record for all model responses, including older Gemini attempts.

In [ ]:
# Column order for the final CSV.
FINAL_COLUMNS = [
    "target_word",
    "sentence_id",
    "sentence",
    "source",
    "generation_model",
    "prompt_version",
    "batch_id",
    "valid_exact_match",
    "exact_match_count",
    "word_count",
    "is_duplicate",
    "valid",
]

# Main output and invalid-row output.
sentences_df = sentences_df[FINAL_COLUMNS].sort_values(
    ["batch_id", "target_word", "sentence_id"]
).reset_index(drop=True)
invalid_df = sentences_df.loc[~sentences_df["valid"]].copy()

sentences_df.to_csv(OUTPUT_PATH, index=False)
invalid_df.to_csv(INVALID_PATH, index=False)

# Short summary statistics for thesis notes.
valid_counts = sentences_df.loc[sentences_df["valid"]].groupby("target_word").size()
words_with_at_least_10_valid = [word for word in target_words if valid_counts.get(word, 0) >= 10]
words_with_fewer_than_10_valid = [word for word in target_words if valid_counts.get(word, 0) < 10]

print("Summary")
print(f"total target words: {len(target_words):,}")
print(f"total generated sentences: {len(sentences_df):,}")
print(f"total valid sentences: {int(sentences_df['valid'].sum()):,}")
print(f"number of words with at least 10 valid sentences: {len(words_with_at_least_10_valid):,}")
print(f"words with fewer than 10 valid sentences: {len(words_with_fewer_than_10_valid):,}")

if words_with_fewer_than_10_valid:
    display(
        pd.DataFrame(
            {
                "target_word": words_with_fewer_than_10_valid,
                "valid_sentence_count": [int(valid_counts.get(word, 0)) for word in words_with_fewer_than_10_valid],
            }
        )
    )

print(f"\nSaved: {OUTPUT_PATH}")
print(f"Saved: {RAW_PATH}")
print(f"Saved: {INVALID_PATH}")